In [ ]:
import sys
import maboss
import pandas as pd
import numpy as np
from pathlib import Path
sys.path.append("/Users/emilieyu/endotehelial-masboss")
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

from src.config import load_sim_config, load_sweep_config, load_spatial_config
from src.paths import *
from boolean_models.scripts import run_perturbations, run_sweeps

sim_cfg = load_sim_config()
sweep_cfg = load_sweep_config()
spatial_cfg = load_spatial_config()

MODELS_BND = DATA_DIR / sim_cfg['model']['bnd_v2']
MODELS_CFG = DATA_DIR / sim_cfg['model']['cfg_v4']

from abm.flow_field import FlowField
from abm.endothelial_cell import EndothelialCell
from abm.membrane_node import MembraneNode

## Recruitment Parameter Table

In [51]:
recr_path = BM_RESULTS_DIR / "param_sweep_Recruitment_Sweep.csv"
recr_raw = pd.read_csv(recr_path).dropna(axis=1).round(2)

var_map = {'$DSP_recruitment': 'p_dsp', 
           '$TJP1_recruitment': 'p_tjp1', 
           '$JCAD_recruitment': 'p_jcad'}

In [92]:
recr_df = recr_raw.copy()
recr_df["p1_name"] = recr_df["p1_name"].map(var_map)
index_df = recr_df.set_index(["p1_name", "p1_value"])[["RhoA", "RhoC"]].reset_index()


interpolators_rhoa = {}
interpolators_rhoc = {}

for p in index_df['p1_name'].unique():
    subset = index_df[index_df['p1_name'] == p].sort_values('p1_value')
    interpolators_rhoa[p] = interp1d(subset['p1_value'], subset['RhoA'], bounds_error=False, fill_value='extrapolate')
    interpolators_rhoc[p] = interp1d(subset['p1_value'], subset['RhoC'], bounds_error=False, fill_value='extrapolate')

index_df

,p1_name,p1_value,RhoA,RhoC
0,p_dsp,0.00,0.25,0.80
1,p_dsp,0.05,0.31,0.78
2,p_dsp,0.10,0.35,0.76
3,p_dsp,0.15,0.40,0.73
4,p_dsp,0.20,0.42,0.72
5,p_dsp,0.25,0.46,0.71
6,p_dsp,0.30,0.48,0.71
7,p_dsp,0.35,0.50,0.70
8,p_dsp,0.40,0.52,0.69
9,p_dsp,0.45,0.54,0.68


In [94]:
def get_rho_probability(p_dsp, p_tjp1, p_jcad):
    vals = {
        "p_dsp": (interpolators_rhoa["p_dsp"](p_dsp), interpolators_rhoc["p_dsp"](p_dsp)), 
        "p_tjp1": (interpolators_rhoa["p_tjp1"](p_tjp1), interpolators_rhoc["p_tjp1"](p_tjp1)), 
        "p_jcad": (interpolators_rhoa["p_jcad"](p_jcad), interpolators_rhoc["p_jcad"](p_jcad))
    }

    p_rhoA = np.mean([v[0] for v in vals.values()])
    p_rhoC = np.mean([v[1] for v in vals.values()])

    return p_rhoA, p_rhoC

## FLOW TO MECHNICAL INPUT FOR MABOSS

In [99]:
def hill(tau, K, n):
    """
    S: Sensitivity input (sheae nmagnitude / force)
    K: Half activation thershold
    n: Hill coefficiet
    """
    return tau**n / (K**n + tau**n)

def get_recruitment(cfg, tau, protein):
    """
    Tension-Based Hill recruitment for protein. 
    """
    hill_params = cfg['hill_params'][protein]

    K = hill_params['K']
    n = hill_params['n']

    return hill(tau, K ,n)

In [ ]:
p_dsp = get_recruitment(cfg, tau_dsp, 'DSP')
p_tjp1 = get_recruitment(cfg, tau_tjp1, 'TJP1')
p_jcad = get_recruitment(cfg, tau_jcad, 'JCAD')
get_rho_probability(p_dsp, p_tjp1, p_jcad)

(np.float64(0.6525925925925925), np.float64(0.5944444444444444))